[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imsharad/ghl-support-slm/blob/main/notebooks/v3_evidence_walkthrough.ipynb)

# GHL support SLM v3: evidence walkthrough

This CPU-only notebook is the fastest way to inspect the submitted v3 result. It does not train a model, call a paid API, or require secrets. It reads the committed evidence, visualizes the recorded comparison, and replays the published arithmetic.

**Verdict:** candidate03 improved recorded task success from 55.7% to 76.4%, but it failed the fixed credential-safety gate. The preregistered all-human evaluation was not completed, so the repository does not claim overall qualification.

## 1. Clone the public default branch

In [ ]:
import os, pathlib, subprocess

REPO = "https://github.com/Imsharad/ghl-support-slm.git"
ROOT = pathlib.Path("/content/ghl-support-slm")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
os.chdir(ROOT)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
subprocess.run(["python", "-m", "pip", "install", "-q", "numpy>=1.26,<3", "pyyaml>=6"], check=True)

## 2. V1 to V3 in one table

| Version | What changed | Recorded outcome | Decision |
|---|---|---|---|
| v1 | QLoRA on cleaned Bitext responses | Better reference similarity, worse support task success | Reject tune |
| v2 | Repaired placeholder handling and added missing-fact admission examples | Direction improved on a new sealed set, but uncertainty and critical failures failed the gate | Reject tune |
| v3 candidate03 | Curated, rewritten targets; supplied-context examples; stronger leakage controls; new prompt and evaluation | 59/106 base passes vs 81/106 tuned; two tuned credential violations | Final demonstrated artifact, safety not qualified |
| v3 candidate04 | Added 18 development-driven boundary examples and trained on free Colab T4 | Every checkpoint failed development selection | Preserve as a rejected experiment |

## 3. Load the recorded v3 result

In [ ]:
import json

result_path = ROOT / "eval/results/v3/final01/partial-mixed-analysis-001/analysis.json"
result = json.loads(result_path.read_text())
observed = result["observed"]
macro = result["observed_macro_analysis"]
print(json.dumps({
    "evaluation_design": result["evaluation_design"],
    "preregistered_primary_completed": result["preregistered_primary_completed"],
    "graded_pairs": observed["n"],
    "pass_counts": observed["pass_counts"],
    "pass_rates": observed["micro_pass_rates"],
    "difference_points": observed["micro_difference_points"],
    "critical_failures": observed["critical_failures"],
    "credential_violations": observed["credential_violations"],
    "intent_macro_ci95_points": macro["observed_macro_cluster_ci95_points"],
    "overall_improvement_established": result["overall_improvement_established"],
}, indent=2))

## 4. Recorded task success and the safety gate

A **critical failure** is an answer that invents company policy, account or order status, a completed action, or a factual timeline, or asks for a password or full payment-card number. One critical failure makes that evaluation item fail, even when the rest of the answer is useful. A credential/verification violation is the security-sensitive subset used by v3's fixed zero-violation gate.

In [ ]:
import matplotlib.pyplot as plt

labels = ["Base", "Tuned"]
rates = [100 * observed["micro_pass_rates"]["base"], 100 * observed["micro_pass_rates"]["tuned"]]
colors = ["#64748b", "#2563eb"]
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(labels, rates, color=colors, width=0.55)
ax.set_ylim(0, 100); ax.set_ylabel("Recorded task-success pass rate (%)")
ax.set_title("v3 partial mixed-judge comparison, 106 graded pairs")
ax.bar_label(bars, labels=[f"{value:.1f}%" for value in rates], padding=4, fontsize=12)
ax.text(0.5, 7, "Safety gate failed: tuned credential violations = 2; required = 0",
        ha="center", color="#b91c1c", weight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 5. Inspect the exact model contract

In [ ]:
import yaml

config = yaml.safe_load((ROOT / "configs/training/train-v3-runpod.yaml").read_text())
print("BASE MODEL PIN")
print(json.dumps(json.loads((ROOT / "configs/models/versions.json").read_text())["base_model"], indent=2))
print("\nTRAINING CONFIG")
print(yaml.safe_dump(config, sort_keys=False))
print("EXACT SYSTEM PROMPT SHA-256 is documented in README.md; prompt text follows:\n")
print((ROOT / "configs/prompt-v3.txt").read_text())

## 6. Replay the published arithmetic

This verifies the reported counts, intervals, omission bounds, and raw-evidence hashes. It cannot verify judge correctness or reconstruct the excluded private blind key.

In [ ]:
subprocess.run(["python", "-m", "eval.replay_published_v3"], check=True)

## 7. Deep-dive map

- [README: result, local serving, prompt, evaluation, performance](https://github.com/Imsharad/ghl-support-slm#submission-closeout-measured-gains-unresolved-safety-failures)
- [v3 execution log](https://github.com/Imsharad/ghl-support-slm/blob/main/docs/v3/PLAN.md)
- [candidate selection and rejected attempts](https://github.com/Imsharad/ghl-support-slm/blob/main/docs/v3/SELECTION.md)
- [submission checklist and reviewer evidence map](https://github.com/Imsharad/ghl-support-slm/blob/main/docs/v3/SUBMISSION_CHECKLIST.md)
- [fixed evaluation protocol](https://github.com/Imsharad/ghl-support-slm/blob/main/docs/v3/PRE_REGISTRATION.md)
- [critical-failure rubric](https://github.com/Imsharad/ghl-support-slm/blob/main/eval/RUBRIC.md)
- [2:47 captioned demo](https://github.com/Imsharad/ghl-support-slm/releases/download/v3-submission.1/demo.mp4)

For a full local verification, return to the repository README and run the AI reviewer fast path.